In [ ]:
from pprint import pprint

import numpy as np
import pandas as pd

In [ ]:
# two lines you will need again and again and again
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 500)

#### Credits
The examples and structure are based heavily on pandas own documentation.
More specific [this](https://pandas.pydata.org/docs/user_guide/dsintro.html) guide.

In [ ]:
d = {"b": 1, "a": 0, "c": 2, "d": 6}

s = pd.Series(d)
print(s)

print()

# pass index with fewer indices - the key "a" from the dict is dropped
s = pd.Series(d, index=[*set(d.keys()) - set("a")])
print(s)

In [ ]:
# keys not in the dict have the value np.NaN
s = pd.Series(d, index=[*d.keys(), "e"], name="fancy_series")  # name is optional
print(s)
print()
print(type(s["e"]))

# accessing a non existing key with .get() returns None, as with dicts
print(s.get("x"))

In [ ]:
print(s)

print()

# basic slicing
print(s[1:])

In [ ]:
s_add = s + s
print(s_add)

In [ ]:
s_add_shift = s[1:] + s[::-1]
print(s_add_shift)  # be careful - the new series is sorted by index (ascending)

print()

print(s_add_shift[1:])

In [ ]:
 # change the name of one index in the same series
s.rename(index={"a": "z"}, inplace=True)
print(s)

print()

# test if index "a" is still in there
print("a" in s)

In [ ]:
# masking 
s_greater_than_median = s[s > s.median()]
print(s_greater_than_median)

In [ ]:
# add an entry
s["f"] = 42
print(s)

In [ ]:
# revert a Series to a list
series_list = s.tolist()
print(series_list)

# DataFrame
The DataFrame is the bigger brother of the Series.
It can be imagines as a table, 2D-Matrix or a dict of Series.

A column of a DataFrame is a Series.

### Creation

In [ ]:
# create a DataFrame from dict of lists (lists must have the same length!)
d = {"one": [1.0, 2.0, 3.0], "two": [4.0, 3.0, 2.0]}

# the column names are determined by the key of the dict
df = pd.DataFrame(d)  # as with indices it's also possible to select only certain columns
print(df)

print()

print(df.index)
print(df.columns)
print(df.shape)  # looks familiar from numpy

In [ ]:
# it's just a better numpy-array
# access the array underneath using .values
vals = df.values
print(vals)
print(type(vals))

In [ ]:
# change type of array just like a numpy
df = df.astype(np.int64)
print(df)

### Accessing a DataFrame
Normal slicing and advanced indexing can both be used

In [ ]:
# select one column
selected = df["one"]
print(selected)
print(type(selected))

print()

# multiply columns like keys
res = df["one"] * df["two"]
# a series to be inserted will be cut down to the indices of the df
# res[42] = 69
# missing cols are filled with np.NaN
# del res[0]

df["three"] = res
print(df)

In [ ]:
# select multiple columns based on their name
selected = df[["one", "two"]]  # if there is only one element in the array it's still a DataFrame that's returned
print(selected)
print(type(selected))

In [ ]:
# select multiple rows based on a condition (like in numpy!)
selected = df[df["one"] >= 2]  # that's boolean masking as we know it!
print(selected)
print(type(selected))

In [ ]:
# select rows based on index-name with .loc
selected = df.loc[0]
print(selected)
print(type(selected))

print()

# access [index, columns]
selected = df.loc[0:1, "two": "three"]
print(selected)

print()

# also possible for multiple rows
selected = df.loc[[0, 2]]
print(selected)
print(type(selected))

In [ ]:
# target only one value - (row, column)
selected = df.at[2, "three"]
print(selected)

print()

# also usable for replacing
df.at[2, "three"] = 69
print(df)

In [ ]:
# be careful with those slicing methods as arithmetic operations may reorder the columns (alphabetically)!
# this could lead to false indexing...

# select multiple rows based on their index
selected = df[1:3]
print(selected)
print(type(selected))

print()

# slice like with numpy with iloc
selected = df.iloc[::2, 0:2]
print(selected)
print(type(selected))

### Creating / deleting new columns

In [ ]:
# create a column with a mask
col_name = "nice_col"
df[col_name] = 69  # of course, we can use variables for the strings
print(df)

# extract / drop a column
three = df.pop(col_name)  # del is also possible

print()

print(df)

In [ ]:
# insert a column at a particular position
df.insert(1, "one_again", df["one"])  # can't be done for existing cols (by default)
print(df)
del df["one_again"]

### Maths
The arithmetic is very similar to numpy.  
In the way that you can add scalars and broadcast-able dataframes / sequences to a dataframe.
Just be careful with potential reorders of columns.

In [ ]:
# alignment of data on arithmetic operations
# both column and index are used for aligning the correct fields
r_df = pd.DataFrame(np.random.randn(5, 2), columns=["A", "B"])
print(df)

r_df2 = pd.DataFrame(np.random.randn(7, 3), columns=["A", "B", "C"])
print(r_df2)


df_sum = r_df + r_df2
print(df_sum)

In [ ]:
# broadcasting row-wise
print(df)

print()

row_0 = df.iloc[0]#[:2]  # note the alphabetical ordering of the columns of the result!
print(row_0)

print()

res = df - row_0
print(res)

In [ ]:
# merging DataFrames
monster = pd.concat([df, df[1:]], axis=0)
print(monster)

print()

row_count = monster.value_counts()
print(row_count)

# pandas also allows SQL-like joins on a column or row with pd.merge():
# https://pandas.pydata.org/docs/user_guide/10min.html#join

In [ ]:
# apply something on every series in a DataFrame
# 'inject' a print and then set 1 if df's sum is even else 0
summed = df.apply(lambda x: print(x) or 1 if x.sum() % 2 == 0 else 0)
print(summed)

### Query on conditions

In [ ]:
# query for columns based one certain conditions
selected = df[(df > 2) & (df % 2 == 0)]
print(selected)

print()

filtered_df = selected.dropna(how='all', axis="columns")  # any and rows are also possible
print(filtered_df)

# we can use .isna()
# to get a boolean mask for where data is None

filled_df = filtered_df.fillna(value=-1)
print(filled_df)

print()

# the old df isn't altered
print(df)

### Let's get fancy
Multiple indices!
Prime example [here](https://pandas.pydata.org/docs/user_guide/advanced.html#advanced-advanced-hierarchical)

In [ ]:
idx_names = [
    ["bar", "bar", "baz", "baz", "foo", "foo", "qux", "qux"],
    ["one", "two", "one", "two", "one", "two", "one", "two"],
]

col_names = ["A", "B", "C", "D"]

# create a multi-index structure
indices = pd.MultiIndex.from_arrays(idx_names, names=["level_1", "level_2"])

# create a dataframe with these indices
array = np.arange(100, 132).reshape((8, 4))
df2 = pd.DataFrame(array, index=indices, columns=col_names)
print(df2)

In [ ]:
selected = df2["A"]
print(selected)

In [ ]:
# access via index
selected = df2.loc["bar"]
print(selected)

print()

selected = df2.loc["bar", "two"]
print(selected)

In [ ]:
# select based on the second index (internally masking is used)
selected = df2.loc[df2.index.get_level_values('level_2') == 'two']
print(selected)
# note that level_2 is still there...

print()

# get rid of the second index
selected = selected.reset_index(level="level_2", drop=True)
print(selected)

In [ ]:
# use cross-section .xs() instead to select data at a particular level of a multi index
selected = df2.xs('two', level='level_2', axis=0)
print(selected)

# xs can't be used for assigning values!

In [ ]:
# interval selection over multiple indices
selected = df2.loc[("bar", "two"):("foo", "one")]
print(selected)

print()

In [ ]:
# changing the types of indices
data = {
    'value': [10, 20, 30, 40, 50, 60],
    'other_value': [100, 200, 300, 400, 500, 600][::-1]
}

index = pd.MultiIndex.from_tuples([('2023-08-01', 'A'), ('2023-08-01', 'B'),
                                  ('2023-08-02', 'A'), ('2023-08-02', 'B'),
                                  ('2023-08-03', 'A'), ('2023-08-03', 'B')],
                                  names=['date', 'category'])

df2 = pd.DataFrame(data, index=index)

print(df2)
print(type(df2.index.levels[0]))
print()

# convert the 'date' index level to datetime data type using .astype()
index_as_date = df2.index.levels[0].astype('datetime64[ns]')
# replace the index
df2.index = df2.index.set_levels(index_as_date, level='date')

# Display the DataFrame with the datetime index
print(df2)
print(type(df2.index.levels[0]))

In [ ]:
not_before = pd.to_datetime('2023-08-02')
selected = df2.loc[df2.index.get_level_values('date') >= not_before]
print(selected)

In [ ]:
# move an index back to a normal column
df2 = df2.reset_index(level='category')
print(df2)

In [ ]:
# and set it back as index
# make sure not to overwrite the index before, but to append it
df2 = df2.set_index("category", append=True)
print(df2)

### Groups / Groupy
Concept:
* Splitting the data into groups based on some criteria.
* Applying a function to each group independently.
* Combining the results into a data structure.

[docs](https://pandas.pydata.org/pandas-docs/stable/user_guide/groupby)


In [ ]:
# Create a sample sales dataset
data_dict = {
    'Salesperson': ['John', 'Alice', 'John', 'Alice', 'Bob', 'Bob'],
    'Product': ['A', 'B', 'A', 'C', 'B', 'C'],
    'Amount': [100, 200, 150, 300, 250, 350]
}

df_grouping = pd.DataFrame(data_dict)

# Group the data by 'Salesperson' and calculate the total sales amount
# the groups object mainly behaves like a dict where each group is its own key
groups = df_grouping.groupby('Salesperson')
print(groups.groups)
sales_by_person = groups['Amount'].sum()

print()

print(sales_by_person)

In [ ]:
speeds = pd.DataFrame(
    [
        ("bird", "Falconiformes", 389.0),
        ("bird", "Psittaciformes", 24.0),
        ("mammal", "Carnivora", 80.2),
        ("mammal", "Primates", np.nan),
        ("mammal", "Carnivora", 58),
    ],
    index=["falcon", "parrot", "lion", "monkey", "leopard"],
    columns=("class", "order", "max_speed"),
)
print(speeds)
print()

In [ ]:
# simple grouping by class
grouped = speeds.groupby("class")
pprint(grouped.groups)
print()

print(grouped.get_group("bird"))
print()

# calculate the mean max-speed of each group
pprint(grouped["max_speed"].mean())

In [ ]:
# group by various columns
grouped = speeds.groupby(["order", "class"])
pprint(grouped.groups)
print()
# iterate with a for loop
for g_name, g_data in grouped:
    print(f"{g_name=}")
    print(f"type: {type(g_data).__name__}")
    print(g_data)
    print()

print()

In [ ]:
# get some feeling for the groups and their values
info = grouped.describe()
print(info)

In [ ]:
# group by a function that returns new labels for the classes
df = pd.DataFrame(
    {
        "A": ["foo", "bar", "foo", "bar", "foo", "bar", "foo", "foo"],
        "B": np.arange(8),
    }
)
# we can
grouped = df.groupby(lambda x: "even" if x % 2 == 0 else "odd")
print(grouped.groups)

### Some other things

In [ ]:
# we can transpose a DataFrame
transposed = df2.T
print(transposed)

In [ ]:
info = df2.describe()
print(info)

In [ ]:
print(df2)

print()

# sort by index
sorted_df2 = df2.sort_index(axis=0, level="category", ascending=True)
print(sorted_df2)

In [ ]:
print(df2)

print()

# sort by values
sort_df2 = df2.sort_values(axis=0, by="value", ascending=False)
print(sort_df2)

In [ ]:
# create a copy
df22 = df2.copy()
print(df2 == df22)  # compare two DataFrames
print()
print(id(df2) == id(df22))
print(df2 is df22)

In [ ]:
# we can get a table as csv
df2_csv = df2.to_csv()
print(df2_csv)

In [ ]:
# we can get a table as LaTeX
df2_ltx = df2.to_latex(float_format="%.1f")
print(df2_ltx)